# human_eval_ui
Rater interface for the blind head-to-head. Reads `human_eval/pairs.jsonl`, which carries no arm label, no side assignment, no GPT score and no repeat marker. Appends one response per comparison to `human_eval/responses.jsonl` and resumes from it, so a session can be closed at any point.

Per comparison: instruction, reference image(s), then image A and image B. Three forced choices with a tie option — instruction adherence, subject consistency, overall preference — plus image quality, which GPT-4.1 does not measure at all and is the leg's own contribution. Error tags are a fixed multi-select; free text appears only when `other` is ticked.

165 comparisons. Roughly 40 per sitting keeps fatigue out of the later ones.

CPU runtime, no compute units, no API calls.

The `PREREG` string below was reworded after the run. `human_eval/prereg.txt` was written from the original wording at 2026-08-15T14:31:26 and is unaltered; the write is guarded by `if not os.path.exists(P)`, and the cell's stored output prints the frozen text. Three divergences: the per-contrast line drops "33/50 needed at p<0.05", the position-bias line narrows "all comparisons" to "the independent primary comparisons", and the closing paragraph is weakened. `prereg.txt` is the registration of record. Asserted in `verify_artifacts` section 11.

## 1. Pre-registration
Written before the first comparison and left unedited. A single rater who is also the author cannot be blinded to the hypotheses, so the analysis is fixed in advance instead.

In [1]:
PREREG = """
Primary: agreement between the rater's overall preference and the sign of the GPT-4.1 Overall
margin on the same row, pooled over the 150 scored pairs, reported against the GPT margin band.
Rows where the rater answers tie, and rows where the GPT margin is exactly zero, are excluded
from the agreement rate and reported as counts.

Secondary, all pre-specified:
    per-contrast win rate over 50 pairs, ties excluded, two-sided binomial on the non-tie decisions
  per-dimension agreement, testing whether the LoRA effect is confined to subject consistency
  error tag counts by contrast, as the qualitative taxonomy
  self-agreement over the 15 repeated pairs, as the single-rater reliability bound
    left-preference rate over the independent primary comparisons, as a position-bias check, expected near 0.5

The human leg tests whether GPT-4.1 ranks as a human ranks. It does not re-measure the arms:
at 50 pairs per contrast only an effect near 66% preference is detectable, while the arms are
already measured at n=488. A null on LoRA and QLoRA is the predicted result and is consistent with the automated judge;
it is not itself evidence of disagreement, and at this n it does not establish equivalence.
"""

from google.colab import drive
import os, json, datetime
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
HE    = f"{DRIVE}/human_eval"
P = f"{HE}/prereg.txt"
# abort only on post-rating evidence: pairs.jsonl and key.jsonl are written by human_eval_build
# before this notebook is first opened, so their presence is the normal first-run state.
_R = f"{HE}/responses.jsonl"
_rated = os.path.exists(_R) and any(l.strip() for l in open(_R, encoding="utf-8"))
if not os.path.exists(P) and _rated:
    raise RuntimeError(
        "prereg.txt is missing but responses.jsonl already holds recorded ratings. Writing a new "
        "pre-registration now would timestamp it after the evaluation and create a false provenance "
        "record. Restore the frozen prereg.txt from the archive before continuing.")
if not os.path.exists(P):
    open(P, "w").write(PREREG.strip() + f"\n\nfrozen {datetime.datetime.now().isoformat(timespec='seconds')}\n")
    print("pre-registration frozen ->", P)
else:
    print("pre-registration already frozen:")
print(open(P).read())

Mounted at /content/drive
pre-registration already frozen:
Primary: agreement between the rater's overall preference and the sign of the GPT-4.1 Overall
margin on the same row, pooled over the 150 scored pairs, reported against the GPT margin band.
Rows where the rater answers tie, and rows where the GPT margin is exactly zero, are excluded
from the agreement rate and reported as counts.

Secondary, all pre-specified:
  per-contrast win rate over 50 pairs, ties excluded, two-sided binomial, 33/50 needed at p<0.05
  per-dimension agreement, testing whether the LoRA effect is confined to subject consistency
  error tag counts by contrast, as the qualitative taxonomy
  self-agreement over the 15 repeated pairs, as the single-rater reliability bound
  left-preference rate over all comparisons, as a position-bias check, expected near 0.5

The human leg tests whether GPT-4.1 ranks as a human ranks. It does not re-measure the arms:
at 50 pairs per contrast only an effect near 66% preference i

## 2. Pairs, references, progress

In [2]:
import tarfile, shutil
RATER     = "S"                       # initials; only matters if the list is ever rated twice
SESSION   = 45                        # comparisons offered before the notebook stops
PAIRS     = f"{HE}/pairs.jsonl"
RESPONSES = f"{HE}/responses.jsonl"
STAGE     = f"{HE}/images"            # neutrally named copies; the arm directories are never opened
IMGS      = f"{HE}/refs"              # only the references these 165 rows use
TAR       = f"{DRIVE}/imgs_subset.tar"

pairs = [json.loads(l) for l in open(PAIRS, encoding="utf-8")]

# The 165 rows reference a few hundred files out of the corpus's 37,063. Copying the archive
# to local disk and extracting all of it needs tens of GB and fails on a runtime that has not
# got them, so the needed members are read straight out of the archive on Drive and cached
# next to the pair list. One pass, once; later sessions find the cache and skip it.
os.makedirs(IMGS, exist_ok=True)
need = sorted({r for p in pairs for r in p["input_images"]})
absent = [r for r in need if not os.path.exists(f"{IMGS}/{r}")]
print(f"references needed {len(need)} | already cached {len(need)-len(absent)}")

if absent:
    assert os.path.exists(TAR), f"archive not found: {TAR}"

    # Member names may carry a prefix the manifest paths do not: `tar -cf x.tar .` stores
    # "./double/...", `tar -cf x.tar imgs` stores "imgs/double/...". Match on the basename
    # and confirm on the path tail, so any prefix resolves. The first few headers are read
    # first and checked against the wanted set: a mismatch is reported before, not after,
    # a sequential pass over tens of GiB.
    from collections import defaultdict
    by_base = defaultdict(list)
    for w in absent: by_base[os.path.basename(w)].append(w)

    def wanted(member_name):
        n = member_name[2:] if member_name.startswith("./") else member_name
        for x in by_base.get(os.path.basename(member_name), []):
            if n == x or n.endswith("/" + x): return x
        return None

    with tarfile.open(TAR) as t:
        sample = []
        for m in t:
            if m.isfile(): sample.append(m.name)
            if len(sample) >= 5: break
    print(f"archive {os.path.getsize(TAR)/2**30:.1f} GiB; member names look like {sample[:2]}")
    assert any("/" in s for s in sample), f"unexpected archive layout: {sample}"

    print(f"extracting {len(absent)} members, one sequential pass")
    want, got = set(absent), 0
    with tarfile.open(TAR) as t:
        for m in t:
            w = wanted(m.name) if m.isfile() else None
            if w and w in want:
                dst = f"{IMGS}/{w}"
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                with open(dst, "wb") as f: shutil.copyfileobj(t.extractfile(m), f)
                want.discard(w); got += 1
                if got % 50 == 0: print(f"  {got}/{len(absent)}")
                if not want: break
    print(f"  extracted {got}")
    if want:
        print(f"  {len(want)} not found in the archive, e.g. {sorted(want)[:3]}")
        print(f"  archive member names: {sample}")

missing = [r for r in need if not os.path.exists(f"{IMGS}/{r}")]
assert not missing, f"{len(missing)} references still missing, first: {missing[:3]}"

def progress():
    done = set()
    if os.path.exists(RESPONSES):
        for l in open(RESPONSES, encoding="utf-8"):
            try: done.add(json.loads(l)["pair_id"])
            except: pass
    return done, [p for p in pairs if p["pair_id"] not in done][:SESSION]

done, todo = progress()
print(f"references {len(need)} cached | comparisons {len(pairs)} | done {len(done)} | this session {len(todo)}")
if not todo:
    print("nothing left to rate; every comparison is recorded")

references needed 182 | already cached 182
references 182 cached | comparisons 165 | done 165 | this session 0
nothing left to rate; every comparison is recorded


## 3. Rate
One click per question, one per error tag, then submit. Order on screen is randomised per pair and never revealed. Re-running this cell resumes from the responses file rather than restarting the session.

In [3]:
import ipywidgets as W
from IPython.display import display, clear_output
from PIL import Image
import io, time
try:
    from google.colab import output as _co; _co.enable_custom_widget_manager()
except Exception:
    pass

DIMS = [("adherence", "Instruction adherence — which image does what the instruction asks?"),
        ("consistency", "Subject consistency — which keeps the reference subject's identity?"),
        ("quality", "Image quality — artifacts, anatomy, coherence. Not judged by GPT-4.1."),
        ("overall", "Overall preference — which image is preferable?")]
TAGS = ["subject missing entirely", "identity / likeness wrong", "colour or material wrong",
        "shape or proportions wrong", "markings, logo or text wrong",
        "scene or action does not match the instruction", "artifacts or image quality",
        "other"]

def png(path, w=380):
    im = Image.open(path).convert("RGB")
    im.thumbnail((w, w))
    b = io.BytesIO(); im.save(b, format="PNG")
    return W.Image(value=b.getvalue(), format="png")

done, todo = progress()          # re-running this cell resumes instead of restarting
state = {"n": 0, "t0": None}
OUT = W.Output()                 # rendering into an Output widget; clear_output inside a
display(OUT)                     # widget callback is not reliable in Colab

def render():
    with OUT:
        clear_output(wait=True)
        _draw()

def _draw():
    if state["n"] >= len(todo):
        print(f"session complete: {state['n']} rated. {len(pairs)-len(done)-state['n']} remain overall.")
        return
    p = todo[state["n"]]; state["t0"] = time.time()
    refs = W.HBox([png(os.path.join(IMGS, r), 190) for r in p["input_images"]])
    imgs = W.HBox([W.VBox([W.HTML("<b>A</b>"), png(f"{STAGE}/{p['left_png']}")]),
                   W.VBox([W.HTML("<b>B</b>"), png(f"{STAGE}/{p['right_png']}")])])
    q = {}
    for k, prompt in DIMS:
        q[k] = W.ToggleButtons(options=[("A", "A"), ("tie", "tie"), ("B", "B")], value=None,
                               description="", layout=W.Layout(width="260px"))
    tagbtn = [W.ToggleButton(description=t, value=False,
                             layout=W.Layout(width="330px", height="30px")) for t in TAGS]
    tags = W.VBox(tagbtn)
    chosen = lambda: [b.description for b in tagbtn if b.value]
    note = W.Textarea(placeholder="only if 'other' is ticked", layout=W.Layout(width="420px", height="56px"))
    submit, skip = W.Button(description="submit", button_style="success"), W.Button(description="skip")
    msg = W.HTML()

    def write(rec):
        with open(RESPONSES, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        state["n"] += 1; render()

    def on_submit(_):
        if any(q[k].value is None for k, _ in DIMS):
            msg.value = "<b style='color:#b00'>answer all four</b>"; return
        if "other" in chosen() and not note.value.strip():
            msg.value = "<b style='color:#b00'>'other' ticked — describe it</b>"; return
        write({"pair_id": p["pair_id"], "seq": p["seq"], "rater": RATER,
               **{k: q[k].value for k, _ in DIMS},
               "tags": chosen(), "note": note.value.strip(),
               "seconds": round(time.time()-state["t0"], 1),
               "at": datetime.datetime.now().isoformat(timespec="seconds")})

    def on_skip(_):
        write({"pair_id": p["pair_id"], "seq": p["seq"], "rater": RATER, "skipped": True,
               "seconds": round(time.time()-state["t0"], 1),
               "at": datetime.datetime.now().isoformat(timespec="seconds")})

    submit.on_click(on_submit); skip.on_click(on_skip)
    display(W.VBox([
        W.HTML(f"<b>{state['n']+1} / {len(todo)}</b> &nbsp; ({len(done)+state['n']} of {len(pairs)} overall)"),
        W.HTML(f"<div style='max-width:820px'><b>Instruction:</b> {p['instruction']}</div>"),
        W.HTML("<b>Reference(s)</b>"), refs, imgs,
        *[W.VBox([W.HTML(prompt), q[k]]) for k, prompt in DIMS],
        W.HTML("<b>What went wrong in the weaker image</b> (click any that apply; leave empty on a tie)"), tags, note,
        W.HBox([submit, skip]), msg]))

render()

Output()

## 4. Progress

In [4]:
import statistics as st
from collections import Counter
recs = [json.loads(l) for l in open(RESPONSES, encoding="utf-8")] if os.path.exists(RESPONSES) else []
rated = [r for r in recs if not r.get("skipped")]
print(f"rated {len(rated)} | skipped {len(recs)-len(rated)} | remaining {len(pairs)-len(recs)}")
if rated:
    print(f"median {st.median([r['seconds'] for r in rated]):.0f}s per comparison, "
          f"{sum(r['seconds'] for r in rated)/60:.0f} min so far")
    for k, _ in DIMS:
        print(f"  {k:12s} {dict(Counter(r[k] for r in rated))}")
    print(f"  left(A) chosen on overall: "
          f"{sum(1 for r in rated if r['overall']=='A')}/{sum(1 for r in rated if r['overall']!='tie')}"
          f"  (rough running side-balance diagnostic, expected near half; the formal"
          f" position-bias test uses primary pairs only and runs in human_eval_check)")
    tg = Counter(t for r in rated for t in r.get("tags", []))
    print("  tags:", dict(tg.most_common()))

rated 165 | skipped 0 | remaining 0
median 138s per comparison, 453 min so far
  adherence    {'B': 9, 'tie': 149, 'A': 7}
  consistency  {'tie': 119, 'B': 30, 'A': 16}
  quality      {'B': 36, 'A': 33, 'tie': 96}
  overall      {'B': 53, 'A': 40, 'tie': 72}
  left(A) chosen on overall: 40/93  (rough running side-balance diagnostic, expected near half; the formal position-bias test uses primary pairs only and runs in human_eval_check)
  tags: {'artifacts or image quality': 60, 'identity / likeness wrong': 36, 'colour or material wrong': 9, 'scene or action does not match the instruction': 8, 'markings, logo or text wrong': 5, 'other': 2, 'shape or proportions wrong': 2}
